In [1]:
import numpy as np
import os
import pandas as pd
import re
from bs4 import BeautifulSoup
import string, time
import json

In [2]:
temp_df = pd.read_csv("../Dataset/IMDB Dataset.csv")

In [3]:
df = temp_df.iloc[:10000]

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df['sentiment'].value_counts()

sentiment
positive    5028
negative    4972
Name: count, dtype: int64

In [6]:
df.duplicated().sum()

np.int64(17)

In [7]:
df.drop_duplicates(inplace=True)

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
from slang import slang_dict
from nltk.corpus import stopwords

In [10]:
def expand_slang(text):
    words = text.split()

    expanded = [
        slang_dict.get(word, word)
        for word in words
    ]

    return " ".join(expanded)

In [11]:
def remove_stopwords(text):
    new_text = []
    
    for word in text.split():
        if word in stopwords.words('english'):
            new_text.append('')
        else:
            new_text.append(word)
        
    x = new_text[:]
    new_text.clear()
    return " ".join(x)

In [12]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = word_tokenize(text)

    lemmas = [
        lemmatizer.lemmatize(word)
        for word in words
        if word.isalpha()
    ]

    return " ".join(lemmas)

In [13]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )
    text = re.sub(r"\s+", " ", text)

    text = expand_slang(text)

    text = remove_stopwords(text)

    text = lemmatize_text(text)

    text = text.lower()

    return text.strip()

In [14]:
df['review'] = df['review'].apply(clean_text)

In [15]:
df

,review,sentiment
0,one reviewer mentioned watching oz episode you...,positive
1,a wonderful little production the filming tech...,positive
2,i thought wonderful way spend time hot summer ...,positive
3,basically there family little boy jake think t...,negative
4,petter matteis love time money visually stunni...,positive
...,...,...
9995,fun entertaining movie wwii german spy julie a...,positive
9996,give break how anyone say good hockey movie i ...,negative
9997,this movie bad movie but watching endless seri...,negative
9998,this movie probably made entertain middle scho...,negative


In [16]:
X = df.iloc[:,0:1]
y = df['sentiment']

In [17]:
X

,review
0,one reviewer mentioned watching oz episode you...
1,a wonderful little production the filming tech...
2,i thought wonderful way spend time hot summer ...
3,basically there family little boy jake think t...
4,petter matteis love time money visually stunni...
...,...
9995,fun entertaining movie wwii german spy julie a...
9996,give break how anyone say good hockey movie i ...
9997,this movie bad movie but watching endless seri...
9998,this movie probably made entertain middle scho...


In [18]:
y

0       positive
1       positive
2       positive
3       negative
4       positive
          ...   
9995    positive
9996    negative
9997    negative
9998    negative
9999    positive
Name: sentiment, Length: 9983, dtype: str

In [19]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(y)

In [20]:
y

array([1, 1, 1, ..., 0, 0, 1], shape=(9983,))

In [21]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 1)

In [22]:
X_train.shape

(7986, 1)

In [23]:
# Applying BoW
from sklearn.feature_extraction.text import CountVectorizer

In [24]:
cv = CountVectorizer()

In [25]:
X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

In [26]:
X_train_bow.shape

(7986, 67376)

In [27]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()

gnb.fit(X_train_bow, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [28]:
y_pred = gnb.predict(X_test_bow)

from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(y_test, y_pred)

0.6519779669504256

In [29]:
confusion_matrix(y_test, y_pred)

array([[705, 247],
       [448, 597]])

In [30]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

rf.fit(X_train_bow, y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test, y_pred)

0.8407611417125689

In [31]:
cv = CountVectorizer(max_features=3000)

X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.fit_transform(X_test['review']).toarray()

rf = RandomForestClassifier()

rf.fit(X_train_bow, y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test, y_pred)

0.5162744116174262

In [36]:
cv = CountVectorizer(ngram_range=(1,2), max_features=1000)

X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.fit_transform(X_test['review']).toarray()

rf = RandomForestClassifier()

rf.fit(X_train_bow, y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test, y_pred)

0.5603405107661492

### TF-IDF

In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [41]:
tfidf = TfidfVectorizer()

In [44]:
X_train_tfidf = tfidf.fit_transform(X_train['review']).toarray()
X_test_tfidf = tfidf.transform(X_test['review']).toarray()

In [45]:
rf = RandomForestClassifier()

rf.fit(X_train_tfidf, y_train)
y_pred = rf.predict(X_test_tfidf)

accuracy_score(y_test, y_pred)

0.8352528793189785

Word2Vec

In [49]:
import gensim

In [50]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [51]:
story = []
for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))

In [52]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2
)

In [53]:
model.build_vocab(story)

In [54]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(5690821, 6220080)

In [55]:
len(model.wv.index_to_key)

32658

In [56]:
def document_vector(doc):
    # remove out-of-vocabulary words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)

In [57]:
document_vector(df['review'].values[0])

array([-0.26533842,  0.3194739 , -0.33475965, -0.00108506,  0.13669392,
       -0.52216285,  0.03414672,  0.6939611 , -0.28829208, -0.02086852,
       -0.07766294, -0.6738567 ,  0.03032145,  0.12735818,  0.0473533 ,
       -0.16544184,  0.21397834, -0.46466145, -0.03237832, -0.8020646 ,
        0.11713256, -0.0162743 ,  0.18144311, -0.44399107, -0.04653596,
       -0.03891829, -0.41337124, -0.05693533, -0.40788183, -0.15645021,
        0.2547191 , -0.0606952 ,  0.3631639 , -0.43188164, -0.07071175,
        0.32525632,  0.09825695, -0.30473855, -0.24719459, -0.52516896,
        0.01803074, -0.53265375, -0.18919761, -0.04978206,  0.4111088 ,
       -0.22303234, -0.414063  ,  0.01254927,  0.29627863,  0.17941016,
        0.07478113, -0.27645326,  0.10814163,  0.0700928 , -0.26978981,
        0.16468991,  0.15404782, -0.02439046, -0.52702314,  0.19943309,
       -0.13209485,  0.15060282,  0.02313056,  0.0279075 , -0.15035301,
        0.32590425,  0.02343151,  0.3474846 , -0.63185406,  0.43